In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install transformers wikipedia newspaper3k GoogleNews pyvis

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 50.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 KB 27.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 KB 64.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 105.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 KB 25.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 KB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 KB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 108.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 KB 33.6 MB/s eta 0:00:00
     ━━━━━━━

In [4]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import math
import torch
import wikipedia
from newspaper import Article, ArticleException
from GoogleNews import GoogleNews
import IPython
from pyvis.network import Network
import webbrowser

In [5]:
tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large")

In [6]:
def extract_relations_from_model_output(text):
    relations = []
    relation, subject, relation, object_ = '', '', '', ''
    text = text.strip()
    current = 'x'
    text_replaced = text.replace("<s>", "").replace("<pad>", "").replace("</s>", "")
    for token in text_replaced.split():
        if token == "<triplet>":
            current = 't'
            if relation != '':
                relations.append({
                    'head': subject.strip(),
                    'type': relation.strip(),
                    'tail': object_.strip()
                })
                relation = ''
            subject = ''
        elif token == "<subj>":
            current = 's'
            if relation != '':
                relations.append({
                    'head': subject.strip(),
                    'type': relation.strip(),
                    'tail': object_.strip()
                })
            object_ = ''
        elif token == "<obj>":
            current = 'o'
            relation = ''
        else:
            if current == 't':
                subject += ' ' + token
            elif current == 's':
                object_ += ' ' + token
            elif current == 'o':
                relation += ' ' + token
    if subject != '' and relation != '' and object_ != '':
        relations.append({
            'head': subject.strip(),
            'type': relation.strip(),
            'tail': object_.strip()
        })
    return relations

In [7]:
class KB():
    def __init__(self):
        self.relations = []

    def are_relations_equal(self, r1, r2):
        return all(r1[attr] == r2[attr] for attr in ["head", "type", "tail"])

    def exists_relation(self, r1):
        return any(self.are_relations_equal(r1, r2) for r2 in self.relations)

    def merge_relations(self, r1):
        r2 = [r for r in self.relations
              if self.are_relations_equal(r1, r)][0]
        spans_to_add = [span for span in r1["meta"]["spans"]
                        if span not in r2["meta"]["spans"]]
        r2["meta"]["spans"] += spans_to_add

    def add_relation(self, r):
        if not self.exists_relation(r):
            self.relations.append(r)
        else:
            self.merge_relations(r)

    def print(self):
        print("Relations:")
        for r in self.relations:
            print(f"  {r}")

In [8]:
def from_text_to_kb(text, span_length=128, verbose=False):
    # tokenize whole text
    inputs = tokenizer([text], return_tensors="pt")

    # compute span boundaries
    num_tokens = len(inputs["input_ids"][0])
    if verbose:
        print(f"Input has {num_tokens} tokens")
    num_spans = math.ceil(num_tokens / span_length)
    if verbose:
        print(f"Input has {num_spans} spans")
    overlap = math.ceil((num_spans * span_length - num_tokens) /
                        max(num_spans - 1, 1))
    spans_boundaries = []
    start = 0
    for i in range(num_spans):
        spans_boundaries.append([start + span_length * i,
                                 start + span_length * (i + 1)])
        start -= overlap
    if verbose:
        print(f"Span boundaries are {spans_boundaries}")

    # transform input with spans
    tensor_ids = [inputs["input_ids"][0][boundary[0]:boundary[1]]
                  for boundary in spans_boundaries]
    tensor_masks = [inputs["attention_mask"][0][boundary[0]:boundary[1]]
                    for boundary in spans_boundaries]
    inputs = {
        "input_ids": torch.stack(tensor_ids),
        "attention_mask": torch.stack(tensor_masks)
    }

    # generate relations
    num_return_sequences = 3
    gen_kwargs = {
        "max_length": 256,
        "length_penalty": 0,
        "num_beams": 3,
        "num_return_sequences": num_return_sequences
    }
    generated_tokens = model.generate(
        **inputs,
        **gen_kwargs,
    )

    # decode relations
    decoded_preds = tokenizer.batch_decode(generated_tokens,
                                           skip_special_tokens=False)

    # create kb
    kb = KB()
    i = 0
    for sentence_pred in decoded_preds:
        current_span_index = i // num_return_sequences
        relations = extract_relations_from_model_output(sentence_pred)
        for relation in relations:
            relation["meta"] = {
                "spans": [spans_boundaries[current_span_index]]
            }
            kb.add_relation(relation)
        i += 1

    return kb

In [9]:
class KB():
    def __init__(self):
        self.entities = {}
        self.relations = []

    def are_relations_equal(self, r1, r2):
        return all(r1[attr] == r2[attr] for attr in ["head", "type", "tail"])

    def exists_relation(self, r1):
        return any(self.are_relations_equal(r1, r2) for r2 in self.relations)

    def merge_relations(self, r1):
        r2 = [r for r in self.relations
              if self.are_relations_equal(r1, r)][0]
        spans_to_add = [span for span in r1["meta"]["spans"]
                        if span not in r2["meta"]["spans"]]
        r2["meta"]["spans"] += spans_to_add

    def get_wikipedia_data(self, candidate_entity):
        try:
            page = wikipedia.page(candidate_entity, auto_suggest=False)
            entity_data = {
                "title": page.title,
                "url": page.url,
                "summary": page.summary
            }
            return entity_data
        except:
            return None

    def add_entity(self, e):
        self.entities[e] = e
        #self.entities[e["title"]] = {k: v for k, v in e.items() if k != "title"}

    def add_relation(self, r):
        # check on wikipedia
        candidate_entities = [r["head"], r["tail"]]
        #entities = [self.get_wikipedia_data(ent) for ent in candidate_entities]
        entities = [ent for ent in candidate_entities]

        # if one entity does not exist, stop
        if any(ent is None for ent in entities):
            return

        # manage new entities
        for e in entities:
            self.add_entity(e)

        # rename relation entities with their wikipedia titles
        #r["head"] = entities[0]
        #r["tail"] = entities[1]

        # manage new relation
        if not self.exists_relation(r):
            self.relations.append(r)
        else:
            self.merge_relations(r)
    def get_relations(self):
       return self.relations
    def get_entities(self):
       return self.entities

    def print(self):
        print("Entities:")
        for e in self.entities:
            print(f"  {e}")
        print("Relations:")
        for r in self.relations:
            print(f"  {r}")


In [16]:
#添加函数

class KB():
    def __init__(self):
        self.entities = {}
        self.relations = []

    def are_relations_equal(self, r1, r2):
        return all(r1[attr] == r2[attr] for attr in ["head", "type", "tail"])

    def exists_relation(self, r1):
        return any(self.are_relations_equal(r1, r2) for r2 in self.relations)

    def merge_relations(self, r1):
        r2 = [r for r in self.relations
              if self.are_relations_equal(r1, r)][0]
        spans_to_add = [span for span in r1["meta"]["spans"]
                        if span not in r2["meta"]["spans"]]
        r2["meta"]["spans"] += spans_to_add

    def get_wikipedia_data(self, candidate_entity):
        try:
            page = wikipedia.page(candidate_entity, auto_suggest=False)
            entity_data = {
                "title": page.title,
                "url": page.url,
                "summary": page.summary
            }
            return entity_data
        except:
            return None

    def add_entity(self, e):
        self.entities[e] = e
        #self.entities[e["title"]] = {k: v for k, v in e.items() if k != "title"}

    def add_relation(self, r):
        # check on wikipedia
        candidate_entities = [r["head"], r["tail"]]
        #entities = [self.get_wikipedia_data(ent) for ent in candidate_entities]
        entities = [ent for ent in candidate_entities]

        # if one entity does not exist, stop
        if any(ent is None for ent in entities):
            return

        # manage new entities
        for e in entities:
            self.add_entity(e)

        # rename relation entities with their wikipedia titles
        #r["head"] = entities[0]
        #r["tail"] = entities[1]

        # manage new relation
        if not self.exists_relation(r):
            self.relations.append(r)
        else:
            self.merge_relations(r)
    def get_relations(self):
       return self.relations
    def get_entities(self):
       return self.entities

    def get_model_input(self):
       self.input={}

       json_dict={}

       input_e=self.get_entities()
       input_e=list(input_e)
       input_r=self.get_relations()#返回所有的三元组
       triples_value_set=set()#声明三元组的集合
       for every_triple in input_r:#遍历所有三元组中的每个三元组
         triples_value_set.add((every_triple['head'],every_triple['type'],every_triple['tail']))
       triples_value_set=list(triples_value_set)
       triples_list=[]
       for i in triples_value_set:
         for j in i:
           triples_list.append(j)

       json_dict={"entities":input_e,"relation":triples_list}

       return json_dict
    


    def print(self):
        print("Entities:")
        for e in self.entities:
            print(f"  {e}")
        print("Relations:")
        for r in self.relations:
            print(f"  {r}")


In [ ]:
text = """
This page provides a description of the Expertiza based OSS project. <link> is an open source project based on <link> framework. Expertiza allows the instructor to create new assignments and customize new or existing assignments. It also allows the instructor to create a list of topics the students can sign up for. Students can form teams in Expertiza to work on various projects and assignments. Students can also peer review other students' submissions. Expertiza supports submission across various document types, including the URLs and wiki pages. The two major tasks for this project were the following: 1. Refactor existing test cases to follow good coding practices. 2. Check coverage of assignment_creation_spec.rb and improve upon it through additional testing. The assignment_creation_spec file contains test cases that ensure proper functionality of an instructor's workflow in regards to the creation of assignments. An instructor, when logged into Expertiza, should be able to create new assignments where each assignment has its own assigned parameters, rubrics, review strategies and due dates. At the start of this project, all logic and handling of testing this instructor workflow was handled within a single file. This single-file implementation leads to problems with maintenance, and such problems within this implementation can be abstracted into one of these forms: 1. File is too long : This file exceeds 250 lines of code, and should not. 2. Block has too many lines : Within the file, the code used to handle a certain operation is implemented with too many subroutines within one block. 3. Similar/Exact block(s) of code found : Across the file, there were duplicate/near-duplicate blocks of code being used. The first refactor that our team did was to extract methods from what otherwise would be duplicate code and large block sizes. The previous team had started this process, but we were able to follow it to completion. We also then moved all of these extracted methods into the helper class assignment_creation_helper.rb. Here is an example of extracted methods that the previous team had created: <code> The following is an example of how we were able to reduce block size and duplicate code with further extraction of methods into assignment_creation_helper.rb: <image> Another major refactor we did was to break up the one large 'describe' block in assignment_creation_spec, which contained all of the tests for that controller. Since assignment creation has many different pages, we created a separate file for testing each page or responsibility. Beyond the creation of the helper file, assignment_creation_spec.rb was broken up into the following files: 1. assignment_creation_dates_spec.rb 2. assignment_creation_deadlines_spec.rb 3. assignment_creation_general_tab_spec.rb 4. assignment_creation_page_spec.rb 5. assignment_creation_participants_spec.rb 6. assignment_creation_review_strategy_spec.rb 7. assignment_creation_rubrics_spec.rb 8. assignment_creation_topics_spec.rb In the relocation of assignment_creation_spec's functionality and creation of new constituent files, additional refactoring was done to ensure that trailing whitespace was deleted, files ended with a final newline character, and other unnecessary characters were removed. All of this was to remove any other problems that code climate detected. There were a few things that code climate detected that we decided did not need to be refactored. The first was a complaint about blocks being too large, even after we divided the one large 'describe' statement into many smaller ones. This is because, in rspec, each test has its own 'it' block, but all of those must be grouped in a larger 'describe' block. I suggest that this be discussed with the teams who may refactor the other test files or have code climate to be configured to instead look just at the 'it' block size. This is an diagram on how otherwise good ruby code may be flagged by code climate: <code> Additionally, code climate was suggesting to change DateTime to Date or Time. As assignments are due not only on a certain day but at a specific time and that this would require a large refactoring of the entire Expertiza system for tracking submission times, we decided it was beyond the scope of this refactor. As part of the refactoring, we fixed a few broken tests in the original assignments_controller_spec.rb and created additional ones to increase coverage. We used TDD to see which tests that were created were breaking and used that to drive our implementation of the controller. Looking at the methods that were covered in assignments_controller.rb we decided to create the following additional test cases for assignment creation. These are additions to the existing assignments_controller_spec.rb. Links to RSpec screencasts for new test cases. 1. <link> 2. <link> 3. <link>. 1. <link>.
"""



kb = from_text_to_kb(text)
kb.get_model_input()

{'entities': ['Expertiza',
  'OSS',
  '<link>',
  'test case',
  'coding',
  'subroutine',
  'code',
  'block',
  'blocks of code',
  'assignment_creation_helper',
  'helper class',
  'class',
  'methods',
  'assignment_creation_describe',
  'controller',
  'assignment_creation_spec',
  'assignment_creation_dates_spec',
  'assignment_creation_general_tab_spec',
  'trailing whitespace',
  'unnecessary characters',
  'character',
  'whitespace',
  'trailing',
  'newline',
  'characters',
  'rspec',
  'test files',
  "'describe' block",
  'DateTime',
  'Time',
  'ruby',
  'Date',
  'tests',
  'TDD',
  'coverage'],
 'relation': ['tests',
  'part of',
  'TDD',
  'subroutine',
  'subclass of',
  'code',
  'assignment_creation_spec',
  'has part',
  'assignment_creation_dates_spec',
  'subroutine',
  'subclass of',
  'blocks of code',
  'whitespace',
  'subclass of',
  'trailing',
  'rspec',
  'instance of',
  'test files',
  '<link>',
  'use',
  'OSS',
  'assignment_creation_spec',
  'has pa

In [ ]:
#一开始产生所有entities和relation的代码

import csv
cnt =0
with open('/content/drive/MyDrive/REBEL_Dataset/AutoReview_summarized_train_row1.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)        
    for row in reader :
        pid = row[0]
        text = row[1].replace('\n', ' ').replace('\r', '')
        with open(f"/content/drive/MyDrive/Entities_Relations/{pid}_output.txt",'w+',encoding='utf-8') as out:
            kb = from_text_to_kb(text,span_length=128,verbose=True)
            e=kb.get_entities()
            for i in e:
              out.write(str(i)+',')
            relation_triple_list=kb.get_relations()
            triple_set=set()
            for relation_triple in relation_triple_list:
              triple_set.add((relation_triple['head'],relation_triple['type'],relation_triple['tail']))
            for unique_triple in triple_set:
              out.write(",".join(unique_triple)+'\n')
                    

Input has 1029 tokens
Input has 9 spans
Span boundaries are [[0, 128], [112, 240], [224, 352], [336, 464], [448, 576], [560, 688], [672, 800], [784, 912], [896, 1024]]


In [ ]:
#尝试将数据集中一行文字的entities和relation写入一个单独的txt文件

import csv
cnt =0
with open('/content/drive/MyDrive/REBEL_Dataset/AutoReview_summarized_train_row1.csv','r',encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)        
    for row in reader :
        pid = row[0]
        text = row[1].replace('\n', ' ').replace('\r', '')
        with open(f"/content/drive/MyDrive/Entities_Relations/{pid}_output.txt",'w+',encoding='utf-8') as out:
            kb = from_text_to_kb(text,span_length=128,verbose=True)
            input_value=kb.get_model_input()
            input_id_value={'id':pid}
            input_id_value.update(input_value)
            out.write(str(input_id_value))
            

Input has 1029 tokens
Input has 9 spans
Span boundaries are [[0, 128], [112, 240], [224, 352], [336, 464], [448, 576], [560, 688], [672, 800], [784, 912], [896, 1024]]


In [ ]:
#4行文字的entities和relation全部写入一个文件txt
import csv
cnt =0
with open('/content/drive/MyDrive/REBEL_Dataset/AutoReview_summarized_train_row4.csv','r',encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)   
    with open(f"/content/drive/MyDrive/Entities_Relations/entities_relations_output.txt",'w+',encoding='utf-8') as out:     
      for row in reader :
          pid = row[0]
          text = row[1].replace('\n', ' ').replace('\r', '')
          kb = from_text_to_kb(text,span_length=128,verbose=True)
          input_value=kb.get_model_input()
          input_id_value={'id':pid}
          input_id_value.update(input_value)
          out.write(str(input_id_value))

Input has 1029 tokens
Input has 9 spans
Span boundaries are [[0, 128], [112, 240], [224, 352], [336, 464], [448, 576], [560, 688], [672, 800], [784, 912], [896, 1024]]
Input has 1026 tokens
Input has 9 spans
Span boundaries are [[0, 128], [112, 240], [224, 352], [336, 464], [448, 576], [560, 688], [672, 800], [784, 912], [896, 1024]]
Input has 1017 tokens
Input has 8 spans
Span boundaries are [[0, 128], [127, 255], [254, 382], [381, 509], [508, 636], [635, 763], [762, 890], [889, 1017]]
Input has 1022 tokens
Input has 8 spans
Span boundaries are [[0, 128], [127, 255], [254, 382], [381, 509], [508, 636], [635, 763], [762, 890], [889, 1017]]


In [ ]:
#4行文字的entities和relation写入一个文件jsonl
import csv
import json

with open('/content/drive/MyDrive/REBEL_Dataset/AutoReview_summarized_train_row4.csv','r',encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)   
    with open(f"/content/drive/MyDrive/Entities_Relations/entities_relations_jsonl.jsonl",'w+',encoding='utf-8') as jsonlfile:     
      for row in reader :
          pid = row[0]
          text = row[1].replace('\n', ' ').replace('\r', '')
          kb = from_text_to_kb(text,span_length=128,verbose=True)
          input_value=kb.get_model_input()
          input_id_value={'id':pid}
          input_id_value.update(input_value)
          jsonlfile.write(json.dumps(input_id_value))

Input has 1029 tokens
Input has 9 spans
Span boundaries are [[0, 128], [112, 240], [224, 352], [336, 464], [448, 576], [560, 688], [672, 800], [784, 912], [896, 1024]]
Input has 1026 tokens
Input has 9 spans
Span boundaries are [[0, 128], [112, 240], [224, 352], [336, 464], [448, 576], [560, 688], [672, 800], [784, 912], [896, 1024]]
Input has 1017 tokens
Input has 8 spans
Span boundaries are [[0, 128], [127, 255], [254, 382], [381, 509], [508, 636], [635, 763], [762, 890], [889, 1017]]
Input has 1022 tokens
Input has 8 spans
Span boundaries are [[0, 128], [127, 255], [254, 382], [381, 509], [508, 636], [635, 763], [762, 890], [889, 1017]]


In [21]:
#将最后的结果变为list,读取4行元数据集文字，打印entities和relation，简单预览一遍
import csv
import json

with open('/content/drive/MyDrive/REBEL_Dataset/AutoReview_summarized_train_row4.csv','r',encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)     
    for row in reader :
      pid = row[0]
      text = row[1].replace('\n', ' ').replace('\r', '')
      kb = from_text_to_kb(text,span_length=128,verbose=True)
      input_value=kb.get_model_input()
      input_id_value={'id':pid}
      input_id_value.update(input_value)
      a=[]
      a.append(input_id_value)
      print(a)


Input has 1029 tokens
Input has 9 spans
Span boundaries are [[0, 128], [112, 240], [224, 352], [336, 464], [448, 576], [560, 688], [672, 800], [784, 912], [896, 1024]]
[{'id': 'E2011', 'entities': ['<link>', 'OSS', 'Expertiza', 'test case', 'coding', 'subroutine', 'code', 'block', 'assignment_creation_helper', 'helper class', 'class', 'block size', 'assignment_creation_describe', 'controller', 'assignment_creation_spec', 'trailing whitespace', 'unnecessary characters', 'character', 'whitespace', 'trailing', 'rspec', 'test files', 'test file', 'DateTime', 'Time', 'Date', 'tests', 'TDD', 'RSpec'], 'relation': ['<link>', 'use', 'OSS', 'test case', 'part of', 'coding', 'controller', 'has part', 'assignment_creation_spec', 'assignment_creation_helper', 'instance of', 'class', 'RSpec', 'use', 'TDD', 'DateTime', 'instance of', 'Time', 'rspec', 'instance of', 'test file', 'DateTime', 'named after', 'Date', 'rspec', 'use', 'test files', 'test case', 'studied by', 'coding', 'whitespace', 'has pa

In [22]:
#将最后的结果变为list,读取4行元数据集文字，将id,entities和relation写入一个文件json
import csv
import json

with open('/content/drive/MyDrive/REBEL_Dataset/AutoReview_summarized_train_row4.csv','r',encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)
    a=[]   
    with open(f"/content/drive/MyDrive/Entities_Relations/3.4_entities_relations_json.json",'w+',encoding='utf-8') as j:     
      for row in reader :
          pid = row[0]
          text = row[1].replace('\n', ' ').replace('\r', '')
          kb = from_text_to_kb(text,span_length=128,verbose=True)
          input_value=kb.get_model_input()
          input_id_value={'id':pid}
          input_id_value.update(input_value)
          a=[]
          a.append(input_id_value)
          j.write(json.dumps(a))

Input has 1029 tokens
Input has 9 spans
Span boundaries are [[0, 128], [112, 240], [224, 352], [336, 464], [448, 576], [560, 688], [672, 800], [784, 912], [896, 1024]]
Input has 1026 tokens
Input has 9 spans
Span boundaries are [[0, 128], [112, 240], [224, 352], [336, 464], [448, 576], [560, 688], [672, 800], [784, 912], [896, 1024]]
Input has 1017 tokens
Input has 8 spans
Span boundaries are [[0, 128], [127, 255], [254, 382], [381, 509], [508, 636], [635, 763], [762, 890], [889, 1017]]
Input has 1022 tokens
Input has 8 spans
Span boundaries are [[0, 128], [127, 255], [254, 382], [381, 509], [508, 636], [635, 763], [762, 890], [889, 1017]]


In [ ]:
#将dataset中所有行数的文本的entities和relation写入一个文件jsonl
import csv
import json

with open('/content/drive/MyDrive/REBEL_Dataset/AutoReview_summarized_train.csv','r',encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)   
    with open(f"/content/drive/MyDrive/Entities_Relations/3.4_all_entities_relations_josn.json",'w+',encoding='utf-8') as j:     
      for row in reader :
          pid = row[0]
          text = row[1].replace('\n', ' ').replace('\r', '')
          kb = from_text_to_kb(text,span_length=128,verbose=True)
          input_value=kb.get_model_input()
          input_id_value={'id':pid}
          input_id_value.update(input_value)
          a=[]
          a.append(input_id_value)
          j.write(json.dumps(a))

Input has 1029 tokens
Input has 9 spans
Span boundaries are [[0, 128], [112, 240], [224, 352], [336, 464], [448, 576], [560, 688], [672, 800], [784, 912], [896, 1024]]
Input has 1021 tokens
Input has 8 spans
Span boundaries are [[0, 128], [127, 255], [254, 382], [381, 509], [508, 636], [635, 763], [762, 890], [889, 1017]]
Input has 1026 tokens
Input has 9 spans
Span boundaries are [[0, 128], [112, 240], [224, 352], [336, 464], [448, 576], [560, 688], [672, 800], [784, 912], [896, 1024]]
Input has 1017 tokens
Input has 8 spans
Span boundaries are [[0, 128], [127, 255], [254, 382], [381, 509], [508, 636], [635, 763], [762, 890], [889, 1017]]
Input has 1022 tokens
Input has 8 spans
Span boundaries are [[0, 128], [127, 255], [254, 382], [381, 509], [508, 636], [635, 763], [762, 890], [889, 1017]]
Input has 1029 tokens
Input has 9 spans
Span boundaries are [[0, 128], [112, 240], [224, 352], [336, 464], [448, 576], [560, 688], [672, 800], [784, 912], [896, 1024]]
Input has 1049 tokens
Input 